In [1]:
# 1. Instalar librerías necesarias (ya que es un cuaderno nuevo)
!pip install biopython pandas

import pandas as pd
from Bio import SeqIO
from google.colab import files
import io

# 2. Celda interactiva para subir los archivos
print("Por favor, sube tus archivos 'similares_WT.fasta' y 'novel_WT.fasta':")
uploaded = files.upload()

# 3. Función para leer los FASTA y convertirlos en DataFrame
def cargar_fasta_a_dataframe(nombre_archivo):
    datos = []
    # Usamos SeqIO de Biopython para leer el formato FASTA
    for record in SeqIO.parse(nombre_archivo, "fasta"):
        datos.append({
            "PDB_ID": record.id,
            "Secuencia": str(record.seq),
            "Longitud": len(record.seq)
        })
    df = pd.DataFrame(datos)
    print(f"¡Éxito! Se cargaron {len(df)} secuencias desde '{nombre_archivo}'.")
    return df

# 4. Procesar los archivos que acabas de subir
df_similar = pd.DataFrame()
df_novel = pd.DataFrame()

if 'similares_WT.fasta' in uploaded:
    df_similar = cargar_fasta_a_dataframe('similares_WT.fasta')
else:
    print("Advertencia: No se encontró 'similares_WT.fasta'")

if 'novel_WT.fasta' in uploaded:
    df_novel = cargar_fasta_a_dataframe('novel_WT.fasta')
else:
    print("Advertencia: No se encontró 'novel_WT.fasta'")

# Mostrar las primeras filas de uno para verificar
print("\nMuestra de las secuencias cargadas (Novel):")
if not df_novel.empty:
    display(df_novel.head())

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 22.1 MB/s eta 0:00:00
Por favor, sube tus archivos 'similares_WT.fasta' y 'novel_WT.fasta':


Saving similares_WT.fasta to similares_WT.fasta
Saving novel_WT.fasta to novel_WT.fasta
¡Éxito! Se cargaron 30 secuencias desde 'similares_WT.fasta'.
¡Éxito! Se cargaron 30 secuencias desde 'novel_WT.fasta'.

Muestra de las secuencias cargadas (Novel):


,PDB_ID,Secuencia,Longitud
0,10JX,GEFLELMRQENAQLISQLRNAVIQDPDENSFYYDLIDNAPDAMVLV...,145
1,7DKK,MHSWSATVDSRSEEAVRAAARRLAERLLAAGISGKIKIEVEANGIK...,89
2,22OC,DPYLSKPHLTTTTTGLGQDTPYNCAPNSLQQALYKLLRVKISEKTL...,193
3,9TAV,SVKSEYAEAAAVGQEAVAVVNTMKAAFQNGDKEAVAQYLARLASLY...,126
4,28TN,MNNLSKIRRRAGLTQRQIAKELNLTAGAICHYENGKRALSIEQCRK...,82


In [2]:
import math
import numpy as np
import pandas as pd

# Diccionario de la Tabla S3 del paper original
MUTATION_MAP = {
    'S': 'W', 'N': 'A', 'K': 'G', 'F': 'T',
    'T': 'F', 'Q': 'G', 'R': 'A', 'M': 'Q',
    'C': 'W', 'Y': 'G', 'W': 'S', 'L': 'N',
    'D': 'W', 'H': 'A', 'A': 'Q', 'I': 'N',
    'E': 'F', 'G': 'K', 'V': 'E', 'P': 'R'
}

def calcular_probabilidades_paper(longitud, indices_excluidos):
    """
    Calcula los pesos exactos usando la fórmula del paper:
    w_i = epsilon + (1 - epsilon) * (1 - |i - (L-1)/2| / ((L-1)/2))
    """
    epsilon = 0.1
    L = longitud
    centro = (L - 1) / 2
    pesos = np.zeros(L)

    for i in range(L):
        if i not in indices_excluidos:
            # Fórmula exacta del paper
            distancia_normalizada = abs(i - centro) / centro
            pesos[i] = epsilon + (1 - epsilon) * (1 - distancia_normalizada)

    # Normalizar para que sumen 1 (requerido por np.random.choice)
    suma_pesos = np.sum(pesos)
    if suma_pesos == 0:
        return pesos # Si ya no quedan posiciones, devuelve ceros
    return pesos / suma_pesos

def generar_variantes_rigurosas(secuencia_original, pdb_id):
    longitud = len(secuencia_original)
    variantes = {f"{pdb_id}_WT": secuencia_original}

    # --- 1. Mutaciones Puntuales (Tus niveles ajustados) ---
    niveles_puntuales = [0.10, 0.40, 0.70]
    sec_actual = list(secuencia_original)
    indices_mutados = set()

    for nivel in niveles_puntuales:
        num_objetivo = math.ceil(longitud * nivel)
        faltan = num_objetivo - len(indices_mutados)

        while faltan > 0 and len(indices_mutados) < longitud:
            probabilidades = calcular_probabilidades_paper(longitud, indices_mutados)
            # Seleccionamos el índice basándonos en las probabilidades calculadas
            idx = np.random.choice(np.arange(longitud), p=probabilidades)

            aa_original = sec_actual[idx]
            sec_actual[idx] = MUTATION_MAP.get(aa_original, 'A')
            indices_mutados.add(idx)
            faltan -= 1

        variantes[f"{pdb_id}_Mut_{int(nivel*100)}"] = "".join(sec_actual)

    # --- 2. Deleciones (Tus niveles ajustados) ---
    niveles_delecion = [0.03, 0.05, 0.10]
    indices_a_borrar = set()

    for nivel in niveles_delecion:
        num_borrar = math.ceil(longitud * nivel)
        faltan = num_borrar - len(indices_a_borrar)

        while faltan > 0 and len(indices_a_borrar) < longitud:
            probabilidades = calcular_probabilidades_paper(longitud, indices_a_borrar)
            idx = np.random.choice(np.arange(longitud), p=probabilidades)
            indices_a_borrar.add(idx)
            faltan -= 1

        sec_del = "".join([secuencia_original[i] for i in range(longitud) if i not in indices_a_borrar])
        variantes[f"{pdb_id}_Del_{int(nivel*100)}"] = sec_del

    return variantes

def procesar_y_guardar_dataset(df, nombre_archivo_salida):
    if df.empty:
        return

    print(f"Generando variantes para {len(df)} secuencias -> {nombre_archivo_salida}...")
    with open(nombre_archivo_salida, 'w') as f:
        for _, row in df.iterrows():
            variantes = generar_variantes_rigurosas(row['Secuencia'], row['PDB_ID'])
            for nombre, sec in variantes.items():
                f.write(f">{nombre}\n{sec}\n")
    print("¡Listo!")

# Ejecutas para tus dos dataframes
procesar_y_guardar_dataset(df_similar, "Experimento_Similares_Reducido.fasta")
procesar_y_guardar_dataset(df_novel, "Experimento_Novel_Reducido.fasta")

Generando variantes para 30 secuencias -> Experimento_Similares_Reducido.fasta...
¡Listo!
Generando variantes para 30 secuencias -> Experimento_Novel_Reducido.fasta...
¡Listo!
